----------------------------------------------------------------------------------------------------

# ***Banco de Dados Relacionais***

----------------------------------------------------------------------------------------------------

## ***1. Fundamentos Teóricos:*** 

Criado por Edgar F. Codd em 1970, o modelo relacional é baseado na teoria matemática de conjuntos e álgebra relacional. Os dados são representados em relações (tabelas) compostas por:

- ***Tuplas*** → linhas (registros)
- ***Atributos*** → colunas (campos)
- ***Domínio*** → conjunto de valores válidos para um atributo

### ***1.1. Álgebra Relacional:***

| Operação       | Símbolo                  | Equivalente SQL                           |
| ----------- | -------------------------- | ---------------------------------- |
| Seleção    | σ | WHERE        |
| Projeção  | π     | SELECT (colunas)                       |
| União      | ∪                  | UNION                        |
| Diferença    | −  | EXCEPT |
| Produto Cartesiano | ×      | CROSS JOIN            |
| Junção | ⋈       | JOIN            |

### ***1.2. Tabelas:*** 

Em um banco dde dados relacional, uma tabela é uma estrutura organizada de linha e coluna, semelhante a uma planilha, como no Excel (mas ele não é um BD).

Cada linha representa um registro distinto e cada coluna representa um tipo de informação, chamado de campo. Por exemplo, uma tabela 'Clientes' pode ter campos como 'ID', 'nome', 'email' e 'telefone'.

----------------------------------------------------------------------------------------------------

## ***2. Modelagem de Dados em Profundidade***

Diagrama Entidade-Relacionamento (DER)

    ┌─────────────┐         ┌─────────────┐         ┌─────────────┐
    │   CLIENTE   │ 1     N │   PEDIDO    │ N     M │   PRODUTO   │
    │─────────────│─────────│─────────────│─────────│─────────────│
    │ id (PK)     │         │ id (PK)     │         │ id (PK)     │
    │ nome        │         │ cliente_id  │         │ nome        │
    │ email       │         │ data        │         │ preco       │
    │ telefone    │         │ status      │         │ estoque     │
    └─────────────┘         └─────────────┘         └─────────────┘
                                    │
                        ┌────────┴────────┐
                        │  ITEM_PEDIDO    │
                        │─────────────────│
                        │ pedido_id (FK)  │
                        │ produto_id (FK) │
                        │ quantidade      │
                        │ preco_unitario  │
                        └─────────────────┘

### ***2.1 Tipos de Relacionamentos***

In [ ]:
-- 1:1 → Usuário tem um Perfil
CREATE TABLE usuarios (
  id    SERIAL PRIMARY KEY,
  login VARCHAR(50) UNIQUE NOT NULL
);

CREATE TABLE perfis (
  id         SERIAL PRIMARY KEY,
  usuario_id INT UNIQUE REFERENCES usuarios(id),
  bio        TEXT,
  avatar_url VARCHAR(255)
);

-- 1:N → Cliente tem vários Pedidos
CREATE TABLE pedidos (
  id         SERIAL PRIMARY KEY,
  cliente_id INT NOT NULL REFERENCES clientes(id),
  data       TIMESTAMP DEFAULT NOW(),
  status     VARCHAR(20) DEFAULT 'pendente'
);

-- N:M → Pedido tem vários Produtos (tabela intermediária)
CREATE TABLE itens_pedido (
  pedido_id       INT REFERENCES pedidos(id),
  produto_id      INT REFERENCES produtos(id),
  quantidade      INT NOT NULL CHECK (quantidade > 0),
  preco_unitario  NUMERIC(10,2) NOT NULL,
  PRIMARY KEY (pedido_id, produto_id)
);

----------------------------------------------------------------------------------------------------

## ***3 Normalização — Do Zero à 5FN***

### ***3.1 Obs.: Tabela NÃO normalizada (0FN)***

    ┌──────┬──────────┬────────────────────────────┬──────────────────┐
    │  ID  │  CLIENTE │         PEDIDOS            │    TELEFONES     │
    ├──────┼──────────┼────────────────────────────┼──────────────────┤
    │  1   │ Ana      │ Pedido#1-Camisa, Pedido#2  │ 9999, 8888       │
    │  2   │ Bruno    │ Pedido#3-Calça             │ 7777             │
    └──────┴──────────┴────────────────────────────┴──────────────────┘

### ***3.2 Primeira Forma Normal (1FN)***

Eliminar grupos repetitivos e garantir atomicidade dos atributos.

In [ ]:
-- Cada coluna deve ter um único valor atômico
CREATE TABLE clientes (
  id    INT PRIMARY KEY,
  nome  VARCHAR(100)
);

CREATE TABLE telefones (
  id         INT PRIMARY KEY,
  cliente_id INT REFERENCES clientes(id),
  numero     VARCHAR(20)  -- um número por linha
);

### ***3.3 Segunda Forma Normal (2FN)***

In [ ]:
-- ❌ Errado: nome_produto depende só de produto_id, não da PK composta
CREATE TABLE itens_pedido_errado (
  pedido_id    INT,
  produto_id   INT,
  nome_produto VARCHAR(100),  -- dependência parcial!
  quantidade   INT,
  PRIMARY KEY (pedido_id, produto_id)
);

-- ✅ Correto: nome_produto fica na tabela produtos
CREATE TABLE itens_pedido (
  pedido_id  INT,
  produto_id INT,
  quantidade INT,
  PRIMARY KEY (pedido_id, produto_id)
);

### ***3.4 Terceira Forma Normal (3FN)***

In [ ]:
-- ❌ Errado: cidade depende de cep, não de cliente_id
CREATE TABLE clientes_errado (
  id     INT PRIMARY KEY,
  nome   VARCHAR(100),
  cep    VARCHAR(8),
  cidade VARCHAR(100)  -- dependência transitiva!
);

-- ✅ Correto: separar CEP em tabela própria
CREATE TABLE ceps (
  cep    VARCHAR(8) PRIMARY KEY,
  cidade VARCHAR(100),
  estado CHAR(2)
);

CREATE TABLE clientes (
  id   INT PRIMARY KEY,
  nome VARCHAR(100),
  cep  VARCHAR(8) REFERENCES ceps(cep)
);

### ***3.5 Forma Normal de Boyce-Codd (BCNF)***

Versão mais rigorosa da 3FN: todo determinante deve ser uma superchave.

### ***3.6 Quarta (4FN) e Quinta (5FN) Formas Normais***

Tratam de dependências multivaloradas e dependências de junção — casos mais raros em modelagens práticas.

----------------------------------------------------------------------------------------------------